Access data programmatically from the Data Ware House with a service account
=========================

Load the configuration for the tableau site to be used by the service account, authenticate and initialize the VizQL client using this configuration:

In [ ]:
from dotenv import dotenv_values
import tableauserverclient as TSC
from vizql_data_service_py import VizQLDataServiceClient
from utils import generate_jwt

# load the settings
settings = dotenv_values('conf/public-test.env')
# create a JSON web token
jwt = generate_jwt(settings['CONNECTED_APP_CLIENT_ID'], settings['CONNECTED_APP_SECRET_ID'], settings['CONNECTED_APP_SECRET_VALUE'], settings['TABLEAU_USERNAME'])
# define the server and authentication method
tableau_auth = TSC.JWTAuth(jwt, site_id=settings['TABLEAU_SITE_ID'])
server = TSC.Server(settings['SERVER_URL'])
# Authenticate
server.auth.sign_in(tableau_auth)

client = VizQLDataServiceClient(settings['SERVER_URL'], server, tableau_auth)

1. Go to [https://tableau.epfl.ch](https://tableau.epfl.ch)
2. Click on "Explore" in the left panel
3. Find the datasource you want and open it by clicking on it
4. Open the datasource details by clicking on the "<b>ⓘ</b>" on the top right, next to the datasource name.
5. The LUID is displayed at the bottom of the datasource details panel:

![Tableau's datasource details panel](doc/img/tableau_get_luid.png)

6. Put the LUID of the datasource in the code bellow:

In [ ]:
datasource_luid = '1ddd4012-e6fb-4e59-b67d-e4465fdd9ed3'

Initialize the datasource for VizQL:

In [ ]:
from vizql_data_service_py import Datasource

datasource = Datasource(datasourceLuid=datasource_luid)

You can now get the metadata of the datasource which describe each field with its name, caption, datatype and possibly the source.

In [ ]:
from json import dumps
from vizql_data_service_py import ReadMetadataRequest, read_metadata

# create the metadata request and retrieve the response
read_metadata_request = ReadMetadataRequest(datasource=datasource)
read_metadata_response = read_metadata.sync(client=client, body=read_metadata_request)

if read_metadata_response is None:
    # if there is an error getting the metadata, print some details
    print(read_metadata.sync_detailed(client=client, body=read_metadata_request))
else:
    # simplify the response and display the metadata
    metadata = [md.model_dump(mode='json', exclude_unset=True) for md in read_metadata_response.data]
    print(f"Fields description: {dumps(metadata, indent=4)}")

From the list of fields, define the ones you want to download.

In [ ]:
from vizql_data_service_py import DimensionField, Query

# list the fields manually
fields_to_get = [
    DimensionField(fieldCaption='IN_Segment origin'),
    DimensionField(fieldCaption='IN_Segment destination'),
    DimensionField(fieldCaption='OUT_DISTANCE_CORRECTED'),
    DimensionField(fieldCaption='OUT_CO2_CORRECTED'),
]    

# or get all fields:
#fields_to_get = [DimensionField(fieldCaption=md.fieldCaption) for md in read_metadata_response.data]

Now query the API to download the data for the list of fields you selected:

In [ ]:
from vizql_data_service_py import Query, QueryRequest, query_datasource

# create the data query and run it to get the data
query_request = QueryRequest(query=Query(fields=fields_to_get), datasource=datasource)
query_response = query_datasource.sync(client=client, body=query_request)

# show the first 100 rows for the selected fields
print(f"Data: {dumps(query_response.data[:100], indent=4)}")